# Chapter 7.5 - Pooling

Pooling is a fixed spatial summarization operation. It has no learned weights, but it strongly shapes what information survives, how large feature maps remain, and how sensitive later layers are to small shifts.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Every required tensor, helper, and model is defined inside this notebook. The cells use small tensors so that the mechanics are visible without turning the chapter into a larger experiment.

## You are done when you can

- explain pooling as an information tradeoff rather than just downsampling
- compute max and average pooling on tiny tensors
- explain why pooling preserves channel count
- show how pooling can discard precise location information
- use global average pooling to collapse spatial dimensions


In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)


## 7.5.0 The Problem This Notebook Solves

Convolution produces feature maps. Those maps can be large, and nearby positions often contain related evidence. Pooling summarizes small spatial neighborhoods to reduce resolution and make later representations less sensitive to tiny movements.

The important word is summarize. Pooling does not learn a detector. It applies a fixed rule:

- max pooling keeps the strongest response
- average pooling keeps the average response

This creates a tradeoff:

```text
less spatial detail
less memory and compute
more tolerance to small shifts
possible loss of exact location information
```

Pooling is therefore not automatically good or bad. It is an architectural choice about which information should survive.

The handoff from earlier sections:

- convolution creates local feature responses
- padding and stride control response-map geometry
- channels store different feature types
- pooling summarizes each feature map spatially


## 7.5.1 Max Pooling and Average Pooling

Max pooling and average pooling answer different questions.

Max pooling asks:

```text
was this feature strongly present anywhere in the window?
```

Average pooling asks:

```text
what was the typical strength of this feature in the window?
```

For sparse detector-like activations, max pooling can preserve a strong local signal. For smoother feature maps, average pooling can preserve broader context. Neither is universally correct; each encodes a different summary.


In [ ]:
window = torch.tensor([[1.0, 3.0], [2.0, 0.0]])

print("max:", window.max())
print("average:", window.mean())

assert window.max().item() == 3.0
assert window.mean().item() == 1.5


The next cell applies both pooling rules over non-overlapping 2 by 2 windows. Read the input as a single feature map. Each output value summarizes one local block of four values.


In [ ]:
X = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)
max_pool = nn.MaxPool2d(kernel_size=2)
avg_pool = nn.AvgPool2d(kernel_size=2)

Y_max = max_pool(X)
Y_avg = avg_pool(X)

print("input:")
print(X[0, 0])
print("max pooled:")
print(Y_max[0, 0])
print("average pooled:")
print(Y_avg[0, 0])

assert shape(Y_max) == (1, 1, 2, 2)
assert shape(Y_avg) == (1, 1, 2, 2)


## 7.5.2 Pooling Uses Window, Padding, and Stride

Pooling has geometry like convolution, but no learned kernel weights. It still has a window size and a stride, so it still changes output shape.

By default, many pooling layers use stride equal to the pooling window size. That creates non-overlapping windows. Setting stride to 1 creates overlapping windows and preserves more spatial resolution.

The theory-level distinction is:

```text
convolution: learned local weighted summary
pooling: fixed local summary
```


In [ ]:
X = torch.arange(9, dtype=torch.float32).reshape(1, 1, 3, 3)
pool_stride_1 = nn.MaxPool2d(kernel_size=2, stride=1)
pool_stride_2 = nn.MaxPool2d(kernel_size=2, stride=2)

Y1 = pool_stride_1(X)
Y2 = pool_stride_2(X)

print("stride 1 shape:", shape(Y1))
print("stride 2 shape:", shape(Y2))
print("stride 1 output:")
print(Y1[0, 0])

assert shape(Y1) == (1, 1, 2, 2)
assert shape(Y2) == (1, 1, 1, 1)


## 7.5.3 Pooling Preserves Channel Count

Pooling summarizes within each channel separately. It does not mix channels and it does not choose new output channels.

This is a major difference from convolution:

```text
convolution can change channel count
pooling usually preserves channel count
```

The reason is conceptual. Each channel is a feature map. Pooling says "summarize where this same feature appears nearby." It does not say "combine this feature with other feature types." Channel mixing is the job of convolution, especially ordinary multi-channel convolution or 1 by 1 convolution.


In [ ]:
X = torch.arange(2 * 4 * 4, dtype=torch.float32).reshape(1, 2, 4, 4)
pool = nn.MaxPool2d(2)
Y = pool(X)

print("input shape:", shape(X))
print("output shape:", shape(Y))

assert shape(Y) == (1, 2, 2, 2)
assert shape(Y)[1] == shape(X)[1]


## 7.5.4 Pooling Can Discard Precise Location

Pooling can make a representation less sensitive to small shifts because several nearby input configurations collapse to the same output. That is useful when the exact pixel location of a feature should not matter much.

But this is also information loss. If two different local arrangements produce the same pooled value, later layers cannot recover which arrangement happened.

The example places the same strong activation at two different locations inside one 2 by 2 pooling window. Max pooling returns the same result for both.

This is the cleanest way to understand the tradeoff:

```text
pooling gives tolerance by discarding detail
```


In [ ]:
A = torch.zeros(1, 1, 4, 4)
B = torch.zeros(1, 1, 4, 4)
A[0, 0, 0, 0] = 5.0
B[0, 0, 1, 1] = 5.0

pool = nn.MaxPool2d(2)
YA = pool(A)
YB = pool(B)

print("pooled A:")
print(YA[0, 0])
print("pooled B:")
print(YB[0, 0])

assert torch.equal(YA, YB)


## 7.5.5 Global Average Pooling Collapses Spatial Dimensions

Global average pooling averages each entire feature map into one value per channel.

That means it intentionally discards exact spatial location at the end of a feature extractor. The resulting vector says, roughly:

```text
how strongly was each feature channel present overall?
```

Modern CNN classifiers often use global average pooling before the final linear layer because it reduces parameters and encourages the classifier to depend on feature presence rather than exact final-map position.

This also creates a graceful handoff to LeNet. Classic LeNet flattens a small spatial map and uses dense layers. Many modern CNNs instead use global average pooling before classification.


In [ ]:
X = torch.randn(3, 8, 6, 6)
pool = nn.AdaptiveAvgPool2d((1, 1))
Y = pool(X)
flattened = Y.flatten(start_dim=1)

print("pooled shape:", shape(Y))
print("flattened shape:", shape(flattened))

assert shape(Y) == (3, 8, 1, 1)
assert shape(flattened) == (3, 8)


## 7.5.6 Break It Deliberately: Pooling Window Too Large

Like convolution, ordinary pooling needs a valid window location. If the pooling window is larger than the feature map and there is no adaptive rule or padding to handle it, there is no output to compute.

This failure reinforces the same geometry discipline from padding and stride:

```text
every spatial operation has a window geometry
the window must fit or be deliberately adapted
```


In [ ]:
pool = nn.MaxPool2d(5)
X = torch.zeros(1, 1, 4, 4)

try:
    pool(X)
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
else:
    raise AssertionError("The pooling window should not fit.")


## 7.5 Checkpoint

Answer these before moving on. Short markdown answers in the notebook are enough; the chapter does not need a separate notes file.

1. Why is pooling best understood as summarization with information loss?
2. How do max pooling and average pooling summarize a window differently?
3. Why does pooling preserve channel count?
4. Why does pooling sometimes help with small shifts?
5. How can pooling help with small shifts while losing exact location?
6. What shape does global average pooling produce before flattening?
